# Falcon — Training on Colab Pro (G4 GPU 100GB VRAM)

This notebook trains the Falcon multi-input Transformer for age & gender estimation.
Select a dataset using the dropdown below, then run all cells.

In [ ]:
# ============================================================
# DATASET SELECTION
# ============================================================
# Options: utk, imdb, lagenda, fairface, adience, agedb, cacd
DATASET_NAME = "lagenda"

# ============================================================
# MODEL CONFIG
# ============================================================
# Variants: falcon_d1_224, falcon_d1_384, falcon_d2_224, falcon_d2_384,
#           falcon_d3_224, falcon_d3_448, falcon_d4_224, falcon_d4_448,
#           falcon_d5_224, falcon_d5_448, falcon_d5_512
MODEL_NAME = "falcon_d5_512"

# ============================================================
# TRAINING HYPERPARAMETERS
# ============================================================
BATCH_SIZE = 256
EPOCHS = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.05
NUM_WORKERS = 8
WITH_PERSONS = True
AGE_MODE = "distribution"
NUM_AGE_BINS = 101

# ============================================================
# DATA LOCATIONS (Google Drive mount)
# ============================================================
# After mounting Drive, place your dataset under:
#   /content/drive/MyDrive/falcon-data/{DATASET_NAME}/
#   ├── images/
#   └── annotations/

import os
DATA_ROOT = f"/content/drive/MyDrive/falcon-data/{DATASET_NAME}"
IMAGE_PATH = os.path.join(DATA_ROOT, "images")
ANNOT_PATH = os.path.join(DATA_ROOT, "annotations")
OUTPUT_DIR = f"/content/drive/MyDrive/falcon-checkpoints/{DATASET_NAME}"

# ============================================================
# PRETRAINED VOLO CHECKPOINT (optional)
# ============================================================
# Falcon extends VOLO. To load a pretrained VOLO backbone, set:
PRETRAINED_CKPT = ""  # e.g. "/content/drive/MyDrive/volo_d5_224.pth.tar"
# Leave empty to train from scratch.

---
## 1. Environment Setup

In [ ]:
import torch, psutil, platform

print(f"Python      : {platform.python_version()}")
print(f"Torch       : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name()}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"CUDA arch   : {torch.cuda.get_arch_list()}")
print(f"RAM         : {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"CPU cores   : {psutil.cpu_count(logical=True)}")

In [ ]:
# Install Falcon and its dependencies
!git clone https://github.com/hoangtung386/Falcon.git /content/Falcon
%cd /content/Falcon

# Install core ML deps first (CUDA-aware)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 --quiet

# Install remaining dependencies
!pip install -r requirements.txt --quiet

# Install the package in editable mode
!pip install -e . --quiet

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

---
## 1.5. Dataset Preparation (tools/)

In [ ]:
# --- tools.download_lagenda: download LAGENDA dataset ---
if DATASET_NAME == "lagenda" and not os.path.isdir(IMAGE_PATH):
    print("LAGENDA dataset not found. Downloading...")
    import gdown, zipfile
    out_dir = DATA_ROOT
    os.makedirs(out_dir, exist_ok=True)
    ids = ["1QXO0NlkABPZT6x1_0Uc2i6KAtdcrpTbG", "1mNYjYFb3MuKg-OL1UISoYsKObMUllbJx"]
    dests = [f"{out_dir}/lagenda_benchmark_images.zip", f"{out_dir}/lagenda_annotation.csv"]
    for file_id, destination in zip(ids, dests):
        url = f"https://drive.google.com/uc?id={file_id}"
        gdown.download(url, destination, quiet=False)
        if destination.endswith(".zip"):
            with zipfile.ZipFile(destination) as zf:
                zf.extractall(f"{out_dir}/")
            os.remove(destination)
    print("LAGENDA dataset downloaded and extracted.")
else:
    print(f"Skipping download (dataset={DATASET_NAME}, images_exist={os.path.isdir(IMAGE_PATH)})")

In [ ]:
# --- tools.dataset_visualization: visualize annotations ---
from tools.dataset_visualization import visualize
from falcon.data.io import read_csv_annotation_file, natural_key
import cv2
from IPython.display import display, Image as IPImage

csv_files = sorted([f for f in os.listdir(ANNOT_PATH) if f.endswith(".csv")], key=natural_key)
if csv_files:
    csv_path = os.path.join(ANNOT_PATH, csv_files[0])
    bboxes_per_image, _ = read_csv_annotation_file(csv_path, IMAGE_PATH)
    print(f"Visualizing up to 3 samples from {csv_files[0]}...")
    for idx, (img_path, bboxes) in enumerate(bboxes_per_image.items()):
        if idx >= 3:
            break
        im = cv2.imread(img_path)
        if im is None:
            continue
        for bbox_info in bboxes:
            label = f"{bbox_info.gender} Age:{bbox_info.age}"
            if bbox_info.has_face_bbox:
                cv2.rectangle(im, (bbox_info.bbox[0], bbox_info.bbox[1]),
                              (bbox_info.bbox[2], bbox_info.bbox[3]), (0,255,0), 2)
                cv2.putText(im, label, (bbox_info.bbox[0], bbox_info.bbox[1]-5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
            if bbox_info.has_person_bbox:
                cv2.rectangle(im, (bbox_info.person_bbox[0], bbox_info.person_bbox[1]),
                              (bbox_info.person_bbox[2], bbox_info.person_bbox[3]), (255,0,0), 2)
        cv2.imwrite("/tmp/viz_sample.jpg", im)
        display(IPImage("/tmp/viz_sample.jpg"))
else:
    print("No CSV annotation files found for visualization.")

In [ ]:
# --- tools.preparation_utils: inspect dataset statistics ---
from tools.preparation_utils import save_annotations, get_main_face
from falcon.data.io import read_csv_annotation_file, PictureInfo

if csv_files:
    bboxes_dict, annot_type = read_csv_annotation_file(
        os.path.join(ANNOT_PATH, csv_files[0]), IMAGE_PATH
    )
    all_infos = [info for infos in bboxes_dict.values() for info in infos]
    genders = [i.gender for i in all_infos if i.gender != "-1"]
    ages = [i.age for i in all_infos if i.age != "-1"]
    print(f"Annotation type: {annot_type.value}")
    print(f"Total annotations: {len(all_infos)}")
    print(f"With gender  : {len(genders)} ({len(set(genders))} unique: {set(genders)})")
    print(f"With age     : {len(ages)} (range: {min(ages)}-{max(ages)})")
    print(f"With face    : {sum(1 for i in all_infos if i.has_face_bbox)}")
    print(f"With person  : {sum(1 for i in all_infos if i.has_person_bbox)}")
    print(f"Annot format : {'face+body' if annot_type.name == 'PERSONS' else 'face-only'}")
else:
    print("No CSV files to analyze.")

---
## 2. Dataset Analysis (falcon.data.io + falcon.data.transforms)

In [ ]:
import os, pandas as pd

print(f"Images dir      : {IMAGE_PATH}")
print(f"Annotations dir : {ANNOT_PATH}")
print(f"Output dir      : {OUTPUT_DIR}")

assert os.path.isdir(IMAGE_PATH), f"Images directory not found: {IMAGE_PATH}"
assert os.path.isdir(ANNOT_PATH), f"Annotations directory not found: {ANNOT_PATH}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Use falcon.data.io: read_csv_annotation_file + natural_key ---
from falcon.data.io import read_csv_annotation_file, natural_key, InputType, get_input_type

csv_files = sorted([f for f in os.listdir(ANNOT_PATH) if f.endswith(".csv")], key=natural_key)
for f in csv_files:
    bboxes, annot_type = read_csv_annotation_file(os.path.join(ANNOT_PATH, f), IMAGE_PATH)
    total_boxes = sum(len(v) for v in bboxes.values())
    with_person = sum(1 for v in bboxes.values() for p in v if p.has_person_bbox)
    with_face = sum(1 for v in bboxes.values() for p in v if p.has_face_bbox)
    has_gt = sum(1 for v in bboxes.values() for p in v if p.has_gt())
    print(f"  → {f}: {len(bboxes)} images, {total_boxes} boxes "
          f"(faces={with_face}, persons={with_person}, with_gt={has_gt})")

# --- Use falcon.data.transforms.metrics: cumulative_score / cumulative_error ---
from falcon.data.transforms import cumulative_score, cumulative_error
import torch
print(f"\nSample metric: CS@5={cumulative_score(torch.tensor([25.0]), torch.tensor([27.0]), 5):.2f}, "
      f"CE@20={cumulative_error(torch.tensor([25.0]), torch.tensor([27.0]), 20):.2f}")

---
## 2.5. Sanity Check (tests/)

In [ ]:
# Run all unit tests to verify the codebase is healthy before training
!python -m pytest tests/ -v --tb=short 2>&1 | head -60

# --- Quick smoke test (inspired by test_losses) ---
from falcon.losses import AgeGenderLoss, OrdinalAgeLoss, WeightedMSE
import torch

loss_fn = AgeGenderLoss(num_age_bins=101, only_age=True)
preds = torch.randn(4, 101)
targets = torch.tensor([[0.2], [0.5], [0.0], [0.8]])
loss = loss_fn(preds, targets)
print(f"AgeGenderLoss (only_age): {loss.item():.4f}  ✓" if loss.item() > 0 else "✗")

loss_fn2 = AgeGenderLoss(num_age_bins=101, only_age=False)
preds2 = torch.randn(4, 103)
targets2 = torch.tensor([[0.2, 0.0], [0.5, 1.0], [0.0, 0.0], [0.8, 1.0]])
loss2 = loss_fn2(preds2, targets2)
print(f"AgeGenderLoss (with gender): {loss2.item():.4f}  ✓" if loss2.item() > 0 else "✗")

# --- Quick smoke test (inspired by test_transforms) ---
from falcon.data.transforms import class_letterbox, box_iou, cumulative_score, cumulative_error
import numpy as np

img = np.zeros((100, 200, 3), dtype=np.uint8)
result = class_letterbox(img, new_shape=(224, 224))
print(f"class_letterbox (100x200→224x224): {result.shape}  ✓")

box1 = torch.tensor([[0, 0, 10, 10]])
box2 = torch.tensor([[5, 5, 15, 15]])
iou_val = box_iou(box1, box2)
print(f"box_iou: {iou_val.item():.4f}  ✓" if 0 < iou_val.item() < 1 else "✗")

pred = torch.tensor([[25.0], [30.0], [35.0]])
gt = torch.tensor([[25.0], [31.0], [34.0]])
cs = cumulative_score(pred, gt, L=2)
print(f"cumulative_score: {cs.item():.4f}  ✓" if 0 < cs.item() <= 1.0 else "✗")

# --- Quick smoke test (inspired by test_config) ---
from falcon.config import ModelConfig
cfg = ModelConfig()
assert cfg.input_size == 224
assert cfg.use_face_crops is True
print(f"ModelConfig defaults: OK  ✓")

# --- Quick smoke test (inspired by test_data_reader) ---
from falcon.data.io import PictureInfo
info = PictureInfo("path", "25", "M", bbox=[0, 0, 100, 100])
assert info.has_face_bbox is True
info.clear_face_bbox()
assert info.bbox == [-1, -1, -1, -1]
print(f"PictureInfo: OK  ✓")

# --- Quick smoke test (inspired by test_misc) ---
from falcon.data.io import InputType, get_input_type
assert get_input_type("http://example.com/photo.jpg") == InputType.IMAGE
assert get_input_type("http://example.com/stream") == InputType.VIDEO_STREAM
print(f"InputType detection: OK  ✓")

# --- Quick smoke test (inspired by test_structures) ---
from falcon.structures import PersonAndFaceCrops
crops = PersonAndFaceCrops()
(bodies_inds, bodies), (faces_inds, faces) = crops.get_faces_with_bodies(
    use_persons=False, use_faces=True,
)
assert len(bodies) == 0 and len(faces) == 0
print(f"PersonAndFaceCrops (empty): OK  ✓")

print("\nAll sanity checks passed! Ready to train.")

---
## 3. Training (falcon.config + falcon.losses + falcon.model + falcon.data.dataset)

In [ ]:
from falcon.config import ModelConfig
from falcon.model.factory import create_model
from falcon.data.dataset import build as build_data
from falcon.losses import AgeGenderLoss, OrdinalAgeLoss, WeightedMSE
from falcon.eval_metrics import Metrics, time_sync
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn as nn
import torch
import logging, os, time

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
_logger = logging.getLogger("train")

device = torch.device("cuda")

In [ ]:
# Determine num_classes based on age mode
if AGE_MODE == "distribution":
    num_classes = 2 + NUM_AGE_BINS
elif AGE_MODE == "ordinal":
    num_classes = NUM_AGE_BINS
else:
    num_classes = 1

# --- falcon.config.ModelConfig ---
config = ModelConfig(
    with_persons_model=WITH_PERSONS,
    use_persons=WITH_PERSONS,
    num_classes=num_classes,
    device=device,
)

print(f"Model      : {MODEL_NAME}")
print(f"Num classes: {num_classes}")
print(f"In channels: {config.in_chans}")
print(f"Input size : {config.input_size}")
print(f"Age mode   : {AGE_MODE}")
print(f"With persons: {WITH_PERSONS}")

In [ ]:
# --- falcon.data.dataset.build (uses dataset, classification, loader, reader) ---
# Build train loader
train_dataset, train_loader = build_data(
    name=DATASET_NAME,
    images_path=IMAGE_PATH,
    annotations_path=ANNOT_PATH,
    split="train",
    model_config=config,
    workers=NUM_WORKERS,
    batch_size=BATCH_SIZE,
)

# Build validation loader if "val" split exists
try:
    val_dataset, val_loader = build_data(
        name=DATASET_NAME,
        images_path=IMAGE_PATH,
        annotations_path=ANNOT_PATH,
        split="val",
        model_config=config,
        workers=NUM_WORKERS,
        batch_size=BATCH_SIZE,
    )
    print(f"Train: {len(train_dataset)} samples | Val: {len(val_dataset)} samples")
except FileNotFoundError:
    val_loader = None
    print(f"Train: {len(train_dataset)} samples | Val: None (no val split)")

print(f"Train loader : {len(train_loader)} batches")

In [ ]:
# --- falcon.model.factory.create_model (uses falcon_model, cross_attention) ---
model = create_model(
    model_name=MODEL_NAME,
    num_classes=num_classes,
    in_chans=config.in_chans,
    pretrained=False,
    checkpoint_path=PRETRAINED_CKPT if PRETRAINED_CKPT else "",
    filter_keys=["fds."] if PRETRAINED_CKPT else None,
)
model = model.to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total / 1e6:.2f}M")
print(f"Trainable params : {trainable / 1e6:.2f}M")

In [ ]:
# --- falcon.losses: pick loss based on age mode ---
if AGE_MODE == "distribution":
    criterion = AgeGenderLoss(num_age_bins=NUM_AGE_BINS)
elif AGE_MODE == "ordinal":
    criterion = OrdinalAgeLoss(num_classes=NUM_AGE_BINS)
else:
    criterion = WeightedMSE()

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = GradScaler()

print(f"Loss      : {type(criterion).__name__}")
print(f"Optimizer : AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"Scheduler : CosineAnnealingLR (T_max={EPOCHS})")
print(f"AMP       : Enabled")

---
## 4. Train & Validate Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, epoch):
    model.train()
    total_loss = 0.0
    n_batches = 0
    start = time.time()

    for batch_idx, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        with autocast():
            outputs = model(inputs)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        n_batches += 1

        if batch_idx % 10 == 0:
            _logger.info(
                f"Epoch {epoch} [{batch_idx}/{len(loader)}] "
                f"Loss: {loss.item():.4f}  "
                f"LR: {optimizer.param_groups[0]['lr']:.2e}"
            )

    avg_loss = total_loss / n_batches
    elapsed = time.time() - start
    _logger.info(f"Epoch {epoch} avg_loss: {avg_loss:.4f}, time: {elapsed:.1f}s")
    return avg_loss

In [ ]:
def validate(model, loader, criterion, epoch):
    """Validate using falcon.eval_metrics.Metrics."""
    model.eval()

    # --- falcon.eval_metrics.Metrics: tracks MAE, CS, CE, gender acc ---
    metrics = Metrics(l_for_cs=5, draw_hist=False)

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(loader):
            preproc_start = time_sync()
            inputs, targets = inputs.to(device), targets.to(device)
            preproc_time = time_sync() - preproc_start

            infer_start = time_sync()
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, targets)
            infer_time = time_sync() - infer_start

            metrics.update_loss(loss, inputs.size(0))
            metrics.update_time(infer_time, preproc_time, inputs.size(0))

            # Parse outputs: [age_dist, gender_logits] for distribution mode
            if AGE_MODE == "distribution":
                age_out, gender_out = outputs[:, :NUM_AGE_BINS], outputs[:, NUM_AGE_BINS:]
                age_target = targets[:, 0:1] if targets.dim() > 1 else targets.unsqueeze(1)
                gender_target = targets[:, 1].long() if targets.dim() > 1 and targets.size(1) > 1 else None
                metrics.update_regression_age_metrics(age_out, age_target)
                if gender_target is not None:
                    metrics.update_gender_accuracy(gender_out, gender_target)
            else:
                age_target = targets[:, 0:1] if targets.dim() > 1 else targets
                metrics.update_regression_age_metrics(outputs, age_target)

            if batch_idx % 10 == 0:
                _logger.info(f"Val {epoch} [{batch_idx}/{len(loader)}] {metrics.get_info_str(inputs.size(0))}")

    results = metrics.get_result()
    _logger.info(f"Val {epoch} — MAE={results['agetop1']:.2f}  "
                 f"CS@5={results['csl']:.4f}  "
                 f"CE@20={results['max_error']:.4f}  "
                 f"Gender Acc={results.get('gendertop1', 0):.2f}%")
    return results

In [ ]:
print("=" * 60)
print(f"Starting training: {DATASET_NAME} / {MODEL_NAME}")
print(f"Batch size: {BATCH_SIZE} | Epochs: {EPOCHS}")
print("=" * 60)

history = {"train_loss": [], "val_mae": [], "val_cs5": [], "val_gender_acc": []}

for epoch in range(EPOCHS):
    loss = train_epoch(model, train_loader, criterion, optimizer, scaler, epoch)
    scheduler.step()
    history["train_loss"].append(loss)

    if val_loader is not None and (epoch + 1) % 5 == 0:
        val_results = validate(model, val_loader, criterion, epoch)
        history["val_mae"].append(val_results["agetop1"])
        history["val_cs5"].append(val_results["csl"])
        history["val_gender_acc"].append(val_results.get("gendertop1", 0))

    if (epoch + 1) % 10 == 0:
        ckpt_path = os.path.join(OUTPUT_DIR, f"checkpoint-{epoch}.pth.tar")
        torch.save(
            {
                "epoch": epoch,
                "state_dict": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "loss": loss,
                "min_age": train_dataset.min_age,
                "max_age": train_dataset.max_age,
                "avg_age": train_dataset.avg_age,
                "no_gender": False,
                "with_persons_model": WITH_PERSONS,
            },
            ckpt_path,
        )
        _logger.info(f"Saved checkpoint: {ckpt_path}")

final_path = os.path.join(OUTPUT_DIR, "checkpoint-final.pth.tar")
torch.save(
    {
        "epoch": EPOCHS - 1,
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "loss": loss,
        "min_age": train_dataset.min_age,
        "max_age": train_dataset.max_age,
        "avg_age": train_dataset.avg_age,
        "no_gender": False,
        "with_persons_model": WITH_PERSONS,
    },
    final_path,
)
_logger.info(f"Training complete. Final model: {final_path}")

---
## 5. Model Export & Validation Metrics

In [ ]:
# --- falcon.eval_metrics.write_results: save metrics to CSV ---
from falcon.eval_metrics import write_results

# Log training history as CSV
history_path = os.path.join(OUTPUT_DIR, "training_history.csv")
write_results(history_path, history, format="csv")
_logger.info(f"Saved training history: {history_path}")

# Display final metrics
import pandas as pd
pd.DataFrame(history).tail()

---
## 6. Inference Demo (falcon.model.inference + falcon.model.yolo_detector + falcon.structures + falcon.data.transforms)

In [ ]:
# --- falcon.model.inference.Falcon: inference wrapper ---
from falcon.model.inference import Falcon, Meta
from falcon.model.yolo_detector import Detector
from falcon.structures import PersonAndFaceResult, AGE_GENDER_TYPE
from falcon.data.transforms import prepare_classification_images, class_letterbox
import cv2, numpy as np

# Quick smoke test on a sample image
sample = next((f for f in os.listdir(IMAGE_PATH) if f.endswith((".jpg",".png",".jpeg"))), None)
if sample:
    img = cv2.imread(os.path.join(IMAGE_PATH, sample))
    resized = class_letterbox(img, new_shape=640)
    print(f"Sample image: {sample} → shape {img.shape} → letterbox {resized.shape}")

    # --- falcon.data.transforms.image.prepare_classification_images ---
    tensor = prepare_classification_images([resized], target_size=224, device=device)
    print(f"Prepared tensor: {tensor.shape}")

    # --- falcon.model.yolo_detector.Detector ---
    detector = Detector(conf_threshold=0.25, iou_threshold=0.45, device="cuda")
    results = detector.run(img)
    print(f"Detected: {len(results[0].boxes)} objects")

    # --- falcon.structures.PersonAndFaceResult ---
    result_wrapper = PersonAndFaceResult(results[0])
    print(f"  Faces : {result_wrapper.n_faces}")
    print(f"  Persons: {result_wrapper.n_persons}")
    print(f"  Total : {result_wrapper.n_objects}")
else:
    print("No sample images found for inference demo.")